In [ ]:
# ==============================================================================
# 10 Academy W9: Time Series Forecasting for Portfolio Management Optimization
# Comprehensive EDA & Initial Modeling Workflow (01_EDA.ipynb)
# ==============================================================================

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure parent directory is accessible for src modular logic imports
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Standardizing plotting aesthetics to a modern dark theme configuration
sns.set_theme(style="darkgrid")
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'text.color': '#c9d1d9',
    'axes.labelcolor': '#8b949e',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#30363d',
    'font.size': 11
})

# ------------------------------------------------------------------------------
# STEP 1: Import Project Modules & Infrastructure Handlers
# ------------------------------------------------------------------------------
try:
    from src.data_loader import download_asset_data
    from src.forecasting import split_data_chronologically, optimize_arima_params, generate_arima_forecast, evaluate_forecast_metrics
    from src.statistics import perform_adf_test, calculate_sharpe_ratio, calculate_var
    print("[SUCCESS] Found all functional modules inside src/ structure.")
except ImportError:
    print("[WARNING] Source modules not completely found. Using fallback standard implementations...")
    
    # Fallback functions if structural modules are not fully deployed yet
    def download_asset_data(tickers, start_date, end_date):
        import yfinance as yf
        return yf.download(tickers, start=start_date, end=end_date)
        
    def split_data_chronologically(df, target_col, split_date):
        return df[df.index < split_date][target_col], df[df.index >= split_date][target_col]

    def perform_adf_test(series, title=""):
        from statsmodels.tsa.stattools import adfuller
        result = adfuller(series.dropna())
        print(f"--- ADF Test: {title} ---")
        print(f"ADF Statistic: {result[0]:.4f}")
        print(f"p-value: {result[1]:.4f}")
        print(f"Critical Values (5%): {result[4]['5%']:.4f}")
        print("Stationary" if result[1] <= 0.05 else "Non-Stationary")

# ------------------------------------------------------------------------------
# STEP 2: Data Extraction Pipeline (Task 1.1)
# ------------------------------------------------------------------------------
print("\n=== Initiating Data Extraction Framework ===")
tickers = ["TSLA", "BND", "SPY"]
raw_data = download_asset_data(tickers, start_date="2015-01-01", end_date="2026-06-30")

# Flatten multi-indexed column mapping from yfinance if present
if isinstance(raw_data.columns, pd.MultiIndex):
    close_prices = raw_data['Adj Close'].copy()
else:
    close_prices = raw_data[[col for col in raw_data.columns if 'Adj Close' in col or 'Close' in col]].copy()

close_prices.columns = tickers
print(f"Extracted Dataset Shape: {close_prices.shape}")

# ------------------------------------------------------------------------------
# STEP 3: Data Cleaning & Transform Operations (Task 1.2)
# ------------------------------------------------------------------------------
print("\n=== Processing Data Cleaning Routines ===")
# Interpolating trading gaps linearly to maintain integrity without forward-leakage bias
cleaned_prices = close_prices.interpolate(method='linear').dropna()

# Derive structural returns matrix
daily_returns = cleaned_prices.pct_change().dropna()

# ------------------------------------------------------------------------------
# STEP 4: Exploratory Data Analysis & Visualizations (Task 1.3 & 1.4)
# ------------------------------------------------------------------------------
print("\n=== Rendering Visualizations ===")
fig, axes = plt.subplots(3, 1, figsize=(14, 18), sharex=False)

# Plot 1: Standard Normalized Closing Trends
for ticker in tickers:
    axes[0].plot(cleaned_prices.index, cleaned_prices[ticker], label=ticker, linewidth=1.8)
axes[0].set_title("Asset Closing Performance Trends (Jan 2015 - Jun 2026)", color="#58a6ff", fontsize=14)
axes[0].set_ylabel("Adjusted Closing Price ($)")
axes[0].legend(facecolor='#161b22', edgecolor='#30363d')

# Plot 2: Daily Percentage Volatility Distributions
axes[1].plot(daily_returns.index, daily_returns['TSLA'], color='#ff7b72', alpha=0.6, label='TSLA Daily Returns')
axes[1].set_title("Tesla (TSLA) Daily Returns Volatility Profiles", color="#58a6ff", fontsize=14)
axes[1].set_ylabel("Percentage Change")
axes[1].legend(facecolor='#161b22', edgecolor='#30363d')

# Plot 3: 20-Day Rolling Standard Deviation Comparison (Volatility Mapping)
for ticker in tickers:
    rolling_vol = daily_returns[ticker].rolling(window=20).std()
    axes[2].plot(rolling_vol.index, rolling_vol, label=f"{ticker} Volatility", linewidth=1.5)
axes[2].set_title("20-Day Window Rolling Volatility Analysis", color="#58a6ff", fontsize=14)
axes[2].set_ylabel("Standard Deviation")
axes[2].legend(facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
os.makedirs("../data/processed", exist_ok=True)
plt.savefig("../data/processed/eda_performance_plots.png", dpi=300, facecolor='#0d1117')
plt.show()

# Outlier Anomaly Discovery Tracking ($|R_t| > 3\sigma$)
print("\n=== Volatility Outlier Detection (Task 1.4) ===")
for ticker in tickers:
    sigma = daily_returns[ticker].std()
    outliers = daily_returns[np.abs(daily_returns[ticker]) > 3 * sigma]
    print(f"-> {ticker} detected {len(outliers)} trading days displaying anomalous returns outside 3 standard deviations.")

# ------------------------------------------------------------------------------
# STEP 5: Stationarity Assessment & Core Diagnostics (Task 1.5)
# ------------------------------------------------------------------------------
print("\n=== Running Statistical Stationarity Mapping ===")
for ticker in tickers:
    print("-" * 50)
    perform_adf_test(cleaned_prices[ticker], title=f"{ticker} Price Series")
    perform_adf_test(daily_returns[ticker], title=f"{ticker} Return Series")

# ------------------------------------------------------------------------------
# STEP 6: Historical Risk Metrics Engine (Task 1.6)
# ------------------------------------------------------------------------------
print("\n=== Calculating Foundational Risk Trackers ===")
for ticker in tickers:
    returns_vector = daily_returns[ticker]
    # Calculating empirical historical 95% Value at Risk
    var_95 = np.percentile(returns_vector, 5)
    # Annualized Sharpe ratio mapping tracking risk premium adjustments (assuming Rf=0)
    sharpe = (returns_vector.mean() / returns_vector.std()) * np.sqrt(252)
    print(f"[{ticker}] Value at Risk (95% Daily VaR): {var_95:.4f} | Annualized Historical Sharpe Ratio: {sharpe:.4f}")

# ------------------------------------------------------------------------------
# STEP 7: Initial Task 2 Modeling Framework (Train/Test Forecast)
# ------------------------------------------------------------------------------
print("\n=== Executing Initial Progress Task 2 Time Series Model ===")
# Set chronological data parameters
target_asset = "TSLA"
split_cutoff = "2025-01-01"

train, test = split_data_chronologically(cleaned_prices, target_col=target_asset, split_date=split_cutoff)
print(f"Train Sequence Observations ({train.index.min().date()} to {train.index.max().date()}): {len(train)}")
print(f"Test Sequence Observations  ({test.index.min().date()} to {test.index.max().date()}): {len(test)}")

# Fitting baseline parameters (Using fallback defaults if auto_arima optimization isn't mapped)
try:
    from pmdarima import auto_arima
    print("Running optimization algorithm grid search...")
    optimal_order = auto_arima(train, seasonal=False, stepwise=True, suppress_warnings=True).order
except Exception:
    optimal_order = (1, 1, 1) # Standard naive random walk baseline assignment

print(f"Fitted Configuration Parameters Identified: ARIMA{optimal_order}")

from statsmodels.tsa.arima.model import ARIMA
model = ARIMA(train, order=optimal_order)
fitted_model = model.fit()

# Generate forecasts over validation parameters length
forecast_values = fitted_model.forecast(steps=len(test))
forecast_series = pd.Series(forecast_values, index=test.index)

# Quantifying performance metric evaluations
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test, forecast_series)
rmse = np.sqrt(mean_squared_error(test, forecast_series))
mape = np.mean(np.abs((test - forecast_series) / test)) * 100

print("\n==================================================")
print("      INITIAL ARIMA MODEL EVALUATION SUMMARY     ")
print("==================================================")
print(f"Mean Absolute Error (MAE) : {mae:.4f}")
print(f"Root Mean Sq. Error (RMSE): {rmse:.4f}")
print(f"Mean Abs. % Error (MAPE)  : {mape:.2f}%")
print("==================================================")
print("\n[SUCCESS] Entire workflow executed. Ready for Interim Submission upload.")